In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
NRLMF — real-application cross-validation script (LOCAL).

Outer CV : 5 trials × 10-fold pairwise CV  (R's doCrossValidationByPairwise)
Inner CV : 3 × 80/20 ShuffleSplit at natural ratio (HP tuning)
Metrics  : AUPR, AUC, F1, Accuracy, Recall, Specificity, Precision

NRLMF pipeline:
  1. Load U, V, Y from staged_dataset.gz  (same as other methods)
  2. Jaccard similarity from U and V
  3. constrNeig → simD, simT              (constrained similarity, computed once)
  4. NRLMF.fix_model per fold/HP combo    (fit latent factors)
  5. Predict via simD @ est_A @ est_B.T @ simT.T → sigmoid
"""

import os
import sys
import gzip
import pickle
import warnings
import logging

import numpy as np

from tqdm import TqdmSynchronisationWarning

warnings.simplefilter("ignore", TqdmSynchronisationWarning)

# single-threaded per HPC process — rpy2/R manages its own threading
# os.environ.setdefault("OMP_NUM_THREADS", "1")
# os.environ.setdefault("MKL_NUM_THREADS", "1")

In [2]:
# ====== user paths ======
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"
DRIMC_PATH = os.path.join(PATH_ROOT, "scripts/methods/DRIMC")

# --- dataset selection (override with env var DATASET) ---
DATASET = os.environ.get("DATASET", "tuberculosis")
PATH_DATA = os.path.join(PATH_ROOT, "datasets/realAnalysis", DATASET)
PATH_OUTPUT = os.path.join(
    PATH_ROOT, "outputs/results/realAnalysis", DATASET, "fold_cv/origin"
)
PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")

for p in (PATH_OUTPUT, PATH_ARCHIVE):
    os.makedirs(p, exist_ok=True)

filenames = {"output": "results_nrlmf"}

In [3]:
# ====== Python imports ======
sys.path.append(PATH_ROOT)

from sgimc.utils import load
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import ParameterGrid, ShuffleSplit
from scipy.special import expit

from PyDTI3.nrlmf import NRLMF

# ====== R setup via rpy2 ======
import rpy2.robjects as robjects

r = robjects.r

r(f'setwd("{DRIMC_PATH}")')

# attach R packages
for pkg in ["matrixcalc", "data.table", "Rcpp", "ROCR", "Bolstad2", "MESS"]:
    r(f"library({pkg})")

# source R helper files (only what's needed)
for r_file in [
    "doCrossValidationByPairwise.R",
    "constrNeig.R",
]:
    r(f'source("{r_file}")')

# bind R functions to Python
doCrossValidationByPairwise_R = r["doCrossValidationByPairwise"]
constrNeig_R = r["constrNeig"]


# ====== scoring helper (matches simulation scripts) ======
def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]

    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN

    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])

    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])

    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)

    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]


# ====== helpers ======
def to_r_matrix(arr):
    return r.matrix(
        robjects.FloatVector(arr.flatten()),
        byrow=True,
        nrow=arr.shape[0],
        ncol=arr.shape[1],
    )


def extract_fold(savedFolds, trial, fold):
    fold_data = savedFolds.rx2(trial + 1).rx2(fold + 1)
    Y_train = np.array(fold_data.rx2(7))
    test_label = np.array(fold_data.rx2(1)).flatten()
    test_row = np.array(fold_data.rx2(3)).flatten().astype(int) - 1
    test_col = np.array(fold_data.rx2(4)).flatten().astype(int) - 1
    known_drug_idx = np.array(fold_data.rx2(5)).flatten().astype(int)
    known_target_idx = np.array(fold_data.rx2(6)).flatten().astype(int)
    return Y_train, test_row, test_col, test_label, known_drug_idx, known_target_idx

R callback write-console: data.table 1.17.8 using 6 threads (see ?getDTthreads).    
R callback write-console: Latest news: r-datatable.com
  


In [4]:
# ====== load dataset ======
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logging.info("Loading dataset: %s", DATASET)

filename_input = os.path.join(PATH_DATA, "cv_data", "staged_dataset.gz")
U, V, Y = load(filename_input)

# densify
U = np.asarray(U.toarray() if hasattr(U, "toarray") else U, dtype=float)
V = np.asarray(V.toarray() if hasattr(V, "toarray") else V, dtype=float)

# adjacency matrix for R's CV function
Y_dense = Y.toarray() if hasattr(Y, "toarray") else np.asarray(Y)
Y_adj = (Y_dense > 0).astype(float)

# ====== build similarity matrices (Jaccard, computed once) ======
logging.info("Computing Jaccard similarity matrices ...")
simD = 1 - pairwise_distances(U, metric="jaccard")
simT = 1 - pairwise_distances(V, metric="jaccard")

# handle NaN from all-zero rows (Jaccard is undefined → set self-sim = 1, rest = 0)
simD = np.nan_to_num(simD, nan=0.0)
simT = np.nan_to_num(simT, nan=0.0)
np.fill_diagonal(simD, 1.0)
np.fill_diagonal(simT, 1.0)

logging.info("simD: %s,  simT: %s", simD.shape, simT.shape)

# apply neighbourhood constraint via constrNeig (K=3, same as simulation)
K_neig = 3
simD_Robj = to_r_matrix(simD)
simT_Robj = to_r_matrix(simT)

logging.info("Applying constrNeig (K=%d) ...", K_neig)
lap = constrNeig_R(simD_Robj, simT_Robj, K=K_neig)
simD_R = lap.rx2("simD")
simT_R = lap.rx2("simT")
simD_np = np.array(simD_R)
simT_np = np.array(simT_R)
logging.info("Constrained simD: %s,  simT: %s", simD_np.shape, simT_np.shape)

# ====== create outer CV folds via R ======
kfold = 10
numSplit = 5
seeds_R = robjects.IntVector([7771, 8367, 22, 1812, 4659])

logging.info(
    "Creating %d trials × %d-fold pairwise CV splits via R ...", numSplit, kfold
)
savedFolds = doCrossValidationByPairwise_R(
    to_r_matrix(Y_adj), kfold=kfold, numSplit=numSplit, seeds=seeds_R
)

2026-04-30 17:21:51,238 INFO: Loading dataset: tuberculosis


2026-04-30 17:21:51,506 INFO: Computing Jaccard similarity matrices ...
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
2026-04-30 17:24:55,675 INFO: simD: (6949, 6949),  simT: (13, 13)
2026-04-30 17:24:57,122 INFO: Applying constrNeig (K=3) ...
2026-04-30 17:25:09,204 INFO: Constrained simD: (6949, 6949),  simT: (13, 13)
2026-04-30 17:25:09,212 INFO: Creating 5 trials × 10-fold pairwise CV splits via R ...


In [5]:
# ====== parameter grid ======
grid_model = ParameterGrid(
    {
        "cfix": [1, 5, 10],
        "K1": [5],
        "K2": [5],
        "num_factors": [33],
        "lambda_d": [0.125, 0.25, 0.5],
        "lambda_t": [0.125, 0.25, 0.5],
        "alpha": [0.25],
        "beta": [0.125],
        "theta": [0.5],
        "max_iter": [100],
    }
)

N_INNER_KFOLD = 10  # inner CV folds — 10-fold matches outer density (~90% 1's in train)
N_INNER_EVAL = 3  # only evaluate first 3 of N_INNER_KFOLD folds to save time

# ====== flatten combinations ======
combos = []
for trial in range(numSplit):
    for fold in range(kfold):
        for i_m, par_mdl in enumerate(grid_model):
            combos.append((trial, fold, i_m, par_mdl))
n_combos = len(combos)

In [ ]:
# ====== local (notebook) run — loops over ALL combos ======
BASE_SEED = int(os.environ.get("BASE_SEED", str(0x0BADCAFE)))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [local] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logging.info("LOCAL mode — running all %d combos", n_combos)

task_results = []

# cache inner CV folds per outer (trial, fold) — shared across HP combos
_inner_folds_cache = {}

# optional tqdm progress bar
try:
    from tqdm.auto import tqdm

    _iter = tqdm(range(n_combos), desc="combos", total=n_combos)
except ImportError:
    _iter = range(n_combos)

# ====== run all combos ======
for combo_idx in _iter:
    trial, fold, i_m, par_mdl = combos[combo_idx]
    logging.info(
        "Running combo_idx=%d (trial=%d, fold=%d, cfix=%s, lambda_d=%s, lambda_t=%s)",
        combo_idx,
        trial,
        fold,
        par_mdl["cfix"],
        par_mdl["lambda_d"],
        par_mdl["lambda_t"],
    )

    model_seed = BASE_SEED + combo_idx

    try:
        # ====== extract fold data ======
        Y_train_dense, test_row, test_col, test_label, _, _ = extract_fold(
            savedFolds, trial, fold
        )

        # ====== fit on full training → test scores ======
        # NRLMF uses 0/1 encoding — Y_train_dense is already 0/1
        W_train = np.ones_like(Y_train_dense)

        model = NRLMF(
            cfix=par_mdl["cfix"],
            K1=par_mdl["K1"],
            K2=par_mdl["K2"],
            num_factors=par_mdl["num_factors"],
            lambda_d=par_mdl["lambda_d"],
            lambda_t=par_mdl["lambda_t"],
            alpha=par_mdl["alpha"],
            beta=par_mdl["beta"],
            theta=par_mdl["theta"],
            max_iter=par_mdl["max_iter"],
        )
        model.fix_model(W_train, Y_train_dense, simD_np, simT_np)
        est_A, est_B = model.U, model.V

        # predict: simD @ est_A @ est_B.T @ simT.T → sigmoid
        prob_full = expit(simD_np @ est_A @ est_B.T @ simT_np.T)
        prob_test = prob_full[test_row, test_col]

        scores_test = get_metrics(
            np.asarray(test_label, dtype=float),
            np.asarray(prob_test, dtype=float),
        )
        d1_test = int(sum(abs(est_A).max(axis=1) > 0))
        d2_test = int(sum(abs(est_B).max(axis=1) > 0))

        # ====== inner CV — pairwise CV on outer training matrix ======
        # Cache inner folds per (trial, fold): same split for all HP combos
        inner_cache_key = (trial, fold)
        if inner_cache_key not in _inner_folds_cache:
            inner_seed = (BASE_SEED + trial * 100 + fold + 31415) % (2**31 - 1)
            logging.info(
                "Creating inner %d-fold CV (evaluating %d) for trial=%d, fold=%d (seed=%d)",
                N_INNER_KFOLD,
                N_INNER_EVAL,
                trial,
                fold,
                inner_seed,
            )
            inner_savedFolds = doCrossValidationByPairwise_R(
                to_r_matrix(Y_train_dense),
                kfold=N_INNER_KFOLD,
                numSplit=1,
                seeds=robjects.IntVector([inner_seed]),
            )
            _inner_folds_cache[inner_cache_key] = inner_savedFolds
        inner_savedFolds = _inner_folds_cache[inner_cache_key]

        for cv in range(N_INNER_EVAL):
            # extract inner fold: Y_inner has validation entries zeroed out
            Y_inner, val_row, val_col, val_label, _, _ = extract_fold(
                inner_savedFolds, 0, cv  # trial=0 since numSplit=1
            )

            W_inner = np.ones_like(Y_inner)

            model_cv = NRLMF(
                cfix=par_mdl["cfix"],
                K1=par_mdl["K1"],
                K2=par_mdl["K2"],
                num_factors=par_mdl["num_factors"],
                lambda_d=par_mdl["lambda_d"],
                lambda_t=par_mdl["lambda_t"],
                alpha=par_mdl["alpha"],
                beta=par_mdl["beta"],
                theta=par_mdl["theta"],
                max_iter=par_mdl["max_iter"],
            )
            model_cv.fix_model(W_inner, Y_inner, simD_np, simT_np)
            est_A_cv, est_B_cv = model_cv.U, model_cv.V

            prob_full_cv = expit(simD_np @ est_A_cv @ est_B_cv.T @ simT_np.T)
            prob_valid = prob_full_cv[val_row, val_col]
            scores_valid = get_metrics(
                np.asarray(val_label, dtype=float),
                np.asarray(prob_valid, dtype=float),
            )
            d1_valid = int(sum(abs(est_A_cv).max(axis=1) > 0))
            d2_valid = int(sum(abs(est_B_cv).max(axis=1) > 0))

            task_results.append(
                {
                    "trial": trial,
                    "fold": fold,
                    "cfix": par_mdl["cfix"],
                    "K1": par_mdl["K1"],
                    "K2": par_mdl["K2"],
                    "num_factors": par_mdl["num_factors"],
                    "lambda_d": par_mdl["lambda_d"],
                    "lambda_t": par_mdl["lambda_t"],
                    "alpha": par_mdl["alpha"],
                    "beta": par_mdl["beta"],
                    "theta": par_mdl["theta"],
                    "max_iter": par_mdl["max_iter"],
                    "cv": int(cv),
                    "val_score": scores_valid,
                    "val_d1": d1_valid,
                    "val_d2": d2_valid,
                    "test_score": scores_test,
                    "test_d1": d1_test,
                    "test_d2": d2_test,
                }
            )

    except Exception as e:
        logging.exception("Error at combo_idx=%d: %s", combo_idx, str(e))

2026-04-30 17:25:17,409 INFO: LOCAL mode — running all 1350 combos
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
combos:   0%|          | 0/1350 [00:00<?, ?it/s]2026-04-30 17:25:17,551 INFO: Running combo_idx=0 (trial=0, fold=0, cfix=1, lambda_d=0.125, lambda_t=0.125)
2026-04-30 17:25:28,387 INFO: Creating inner 10-fold CV (evaluating 3) for trial=0, fold=0 (seed=195970485)
combos:   0%|          | 4/1350 [03:03<17:01:38, 45.54s/it]2026-04-30 17:28:20,945 INFO: Running combo_idx=4 (trial=0, fold=0, cfix=1, lambda_d=0.25, lambda_t=0.25)


In [5]:
# ====== save results ======
outfile = os.path.join(PATH_OUTPUT, f"{filenames['output']}.gz")

# archive any existing file so we don't silently overwrite a previous run
if os.path.exists(outfile):
    import time as _time

    mdttm = _time.strftime("%Y%m%d_%H%M%S")
    archived = os.path.join(PATH_ARCHIVE, f"{mdttm}_{os.path.basename(outfile)}")
    os.rename(outfile, archived)
    logging.info("Archived existing output to %s", archived)

logging.info("Saving %d result rows to %s", len(task_results), outfile)
with gzip.open(outfile, "wb+", 4) as fout:
    pickle.dump(task_results, fout)

logging.info("Local run finished.")

2026-04-25 17:23:30,382 INFO: Archived existing output to /Users/sijianfan/projects/BiSSGL/outputs/results/realAnalysis/cdataset/fold_cv/origin/archived/20260425_172330_results_nrlmf.gz
2026-04-25 17:23:30,382 INFO: Saving 7200 result rows to /Users/sijianfan/projects/BiSSGL/outputs/results/realAnalysis/cdataset/fold_cv/origin/results_nrlmf.gz
2026-04-25 17:23:30,490 INFO: Local run finished.
